# MMD Base 1B — Google Colab interactive notebook
Start this notebook manually in Colab. Upload a licensed Persian/English UTF-8 corpus.txt and matching provenance.json. This is not an always-on worker.

In [ ]:
!git clone --depth 1 --branch feat/mmd-cloud-training-mct-02 https://github.com/pctaha501-create/MMD-GameForge.git /content/MMD-GameForge
%pip -q install tokenizers bitsandbytes
from google.colab import files
import pathlib, shutil
uploaded = files.upload()  # corpus.txt, provenance.json, and optionally mmd-cloud-resume.zip
DATASET = pathlib.Path('/content/corpus.txt')
PROVENANCE = pathlib.Path('/content/provenance.json')
if not DATASET.exists() or not PROVENANCE.exists(): raise RuntimeError('Upload files named corpus.txt and provenance.json')
REPO = pathlib.Path('/content/MMD-GameForge')
OUTPUT = pathlib.Path('/content/mmd-cloud-run')
resume_zip = pathlib.Path('/content/mmd-cloud-resume.zip')
if resume_zip.exists(): shutil.unpack_archive(str(resume_zip), str(OUTPUT))
import torch
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
if not torch.cuda.is_available(): raise RuntimeError('In Colab, select Runtime > Change runtime type > T4 GPU (or another CUDA GPU).')
gpu_gib = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f'GPU memory: {gpu_gib:.2f} GiB; required preflight: 24 GiB')
if gpu_gib < 24: raise RuntimeError('MMD Base 1B requires at least 24 GiB GPU memory for this preflight. No CPU or paid fallback.')


In [ ]:
!python /content/MMD-GameForge/cloud/runtime/train.py --config /content/MMD-GameForge/cloud/configs/mmd-base-1b-bilingual.json --dataset "$DATASET" --provenance "$PROVENANCE" --output "$OUTPUT"


In [ ]:
import json, pathlib, subprocess, sys, shutil
metrics = json.loads((OUTPUT / 'metrics.json').read_text(encoding='utf-8'))
if not metrics.get('training_complete'):
    resume_zip = shutil.make_archive('/content/mmd-cloud-resume', 'zip', OUTPUT)
    print('Training is incomplete. Download this checkpoint archive, then upload it next time as mmd-cloud-resume.zip:', resume_zip)
    files.download(resume_zip)
    raise RuntimeError(f"Training paused at step {metrics.get('global_step')}/{metrics.get('max_steps')}; no release bundle was created.")
EVALUATION = OUTPUT / 'evaluation.json'
CONFIG = REPO / 'cloud/configs/mmd-base-1b-bilingual.json'
subprocess.run([sys.executable, str(REPO / 'cloud/runtime/evaluate.py'), '--config', str(CONFIG), '--dataset', str(DATASET), '--provenance', str(PROVENANCE), '--checkpoint', str(OUTPUT / 'checkpoint.pt'), '--tokenizer', str(OUTPUT / 'tokenizer.json'), '--output', str(EVALUATION)], check=True)
BUNDLE = pathlib.Path('/content/mmd-base-1b-bilingual-bundle.zip')
subprocess.run([sys.executable, str(REPO / 'cloud/runtime/bundle.py'), '--checkpoint-dir', str(OUTPUT), '--config', str(CONFIG), '--evaluation', str(EVALUATION), '--output', str(BUNDLE)], check=True)
print('Verified model bundle:', BUNDLE)
files.download(str(BUNDLE))
